### Setup and Installation

First, let's install the necessary libraries for this assignment, including `nltk` for natural language processing, `scikit-learn` for machine learning, and `pandas` for data manipulation.

In [17]:
!pip install nltk scikit-learn pandas

### Download NLTK Data

NLTK requires certain datasets for its functions, such as tokenizers and stopwords. Let's download them.

In [18]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### 1. Data Loading

We will use the 20 Newsgroups dataset for this text classification task. It's a collection of approximately 20,000 newsgroup documents, partitioned (nearly) evenly across 20 different newsgroups. We will load a subset of it.

In [19]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd

# Load the 20 Newsgroups dataset
# We'll fetch a subset of categories for simplicity, feel free to adjust
categories = ['alt.atheism', 'soc.religion.christian',
              'comp.graphics', 'sci.med']

newsgroups_train = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)
newsgroups_test = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)

# Create DataFrames for easier manipulation
df_train = pd.DataFrame({'text': newsgroups_train.data, 'target': newsgroups_train.target})
df_test = pd.DataFrame({'text': newsgroups_test.data, 'target': newsgroups_test.target})

# Map target integers to target names for readability
target_names = newsgroups_train.target_names
df_train['target_name'] = df_train['target'].apply(lambda x: target_names[x])
df_test['target_name'] = df_test['target'].apply(lambda x: target_names[x])

print(f"Training data size: {len(df_train)} samples")
print(f"Test data size: {len(df_test)} samples")
print("Categories:", target_names)

print("\nFirst 5 rows of training data:")
display(df_train.head())

Training data size: 2257 samples
Test data size: 1502 samples
Categories: ['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']

First 5 rows of training data:


,text,target,target_name
0,From: sd345@city.ac.uk (Michael Collier)\nSubj...,1,comp.graphics
1,From: ani@ms.uky.edu (Aniruddha B. Deglurkar)\...,1,comp.graphics
2,From: djohnson@cs.ucsd.edu (Darin Johnson)\nSu...,3,soc.religion.christian
3,From: s0612596@let.rug.nl (M.M. Zwart)\nSubjec...,3,soc.religion.christian
4,From: stanly@grok11.columbiasc.ncr.com (stanly...,3,soc.religion.christian


### 2. NLP Preprocessing

In this section, we will apply several NLP preprocessing techniques to clean and prepare our text data for machine learning. This typically involves:

*   **Tokenization**: Breaking down text into individual words or tokens.
*   **Stopword Removal**: Eliminating common words that don't carry much meaning (e.g., 'the', 'is', 'a').
*   **Lemmatization/Stemming**: Reducing words to their base or root form to reduce vocabulary size and normalize text.

Let's define a preprocessing function and apply it to our text data.

In [20]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import re

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-z]', ' ', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and lemmatize
    processed_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    return ' '.join(processed_tokens)

# Apply preprocessing to training and test data
df_train['processed_text'] = df_train['text'].apply(preprocess_text)
df_test['processed_text'] = df_test['text'].apply(preprocess_text)

print("First 5 rows of training data with processed text:")
display(df_train[['text', 'processed_text', 'target_name']].head())

First 5 rows of training data with processed text:


,text,processed_text,target_name
0,From: sd345@city.ac.uk (Michael Collier)\nSubj...,city michael collier subject converting image ...,comp.graphics
1,From: ani@ms.uky.edu (Aniruddha B. Deglurkar)\...,ani uky edu aniruddha deglurkar subject help s...,comp.graphics
2,From: djohnson@cs.ucsd.edu (Darin Johnson)\nSu...,djohnson ucsd edu darin johnson subject harras...,soc.religion.christian
3,From: s0612596@let.rug.nl (M.M. Zwart)\nSubjec...,let rug zwart subject catholic church poland o...,soc.religion.christian
4,From: stanly@grok11.columbiasc.ncr.com (stanly...,stanly grok columbiasc ncr com stanly subject ...,soc.religion.christian


### 3. Text Vectorization

After preprocessing, text data needs to be converted into numerical representations that machine learning algorithms can understand. This process is called text vectorization. We will use two common techniques:

*   **CountVectorizer**: Converts a collection of text documents to a matrix of token counts.
*   **TF-IDF Vectorizer**: Transforms text into a matrix where each cell represents the TF-IDF score of a word, reflecting its importance in a document relative to the entire corpus.

Let's apply the TF-IDF Vectorizer to our processed text data.

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features to 5000 for practicality

# Fit and transform the training data
X_train_tfidf = tfidf_vectorizer.fit_transform(df_train['processed_text'])

# Transform the test data
X_test_tfidf = tfidf_vectorizer.transform(df_test['processed_text'])

y_train = df_train['target']
y_test = df_test['target']

print(f"TF-IDF vectorized training data shape: {X_train_tfidf.shape}")
print(f"TF-IDF vectorized test data shape: {X_test_tfidf.shape}")

TF-IDF vectorized training data shape: (2257, 5000)
TF-IDF vectorized test data shape: (1502, 5000)


### 4. Model Building and Training

Now that our text data is vectorized, we can train a machine learning model for classification. We will use a Support Vector Machine (SVM) with a linear kernel (`LinearSVC`), a popular choice for text classification due to its effectiveness with high-dimensional data.

In [22]:
from sklearn.svm import LinearSVC

# Initialize and train the LinearSVC model
model = LinearSVC(random_state=42)
model.fit(X_train_tfidf, y_train)

print("Model training complete.")

Model training complete.


### 5. Model Evaluation

After training, it's crucial to evaluate the model's performance on unseen data (the test set). We will predict the labels for the test data and then calculate the accuracy score and a detailed classification report.

### Steps to Upload Your Project to GitHub from Colab

1.  **Create a New GitHub Repository**: Go to [GitHub](https://github.com/) and create a *new, empty repository*. Do NOT initialize it with a README, .gitignore, or license, as we will push our existing Colab project to it.
2.  **Get your GitHub Repository URL**: Once created, copy the URL of your new GitHub repository (e.g., `https://github.com/your-username/your-repo-name.git`).
3.  **Generate a Personal Access Token (PAT)**: If you haven't already, you'll need a [GitHub Personal Access Token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/creating-a-personal-access-token) to authenticate from Colab. Make sure it has `repo` scope permissions.
4.  **Execute the following commands in Colab**: Replace `YOUR_GITHUB_REPOSITORY_URL` with your actual repository URL and `YOUR_GITHUB_TOKEN` with your PAT.

In [24]:
#@title 1. Configure Git (replace with your name and email)
!git config --global user.name "Your Name"
!git config --global user.email "your_email@example.com"
print("Git configured with your name and email.")

Git configured with your name and email.


In [25]:
#@title 2. Initialize a Git repository in your Colab environment
# This creates a hidden .git directory in the current working directory
!git init
print("Git repository initialized.")

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
Git repository initialized.


In [26]:
#@title 3. Add all project files to the repository
# This adds all files in the current directory and its subdirectories
!git add .
print("All project files added to the staging area.")

All project files added to the staging area.


In [27]:
#@title 4. Commit the changes
# This saves the changes to the local repository history
!git commit -m "Initial project commit from Colab"
print("Changes committed to the local repository.")

[master (root-commit) 6e85a76] Initial project commit from Colab
 21 files changed, 51059 insertions(+)
 create mode 100644 .config/.last_opt_in_prompt.yaml
 create mode 100644 .config/.last_survey_prompt.yaml
 create mode 100644 .config/.last_update_check.json
 create mode 100644 .config/active_config
 create mode 100644 .config/config_sentinel
 create mode 100644 .config/configurations/config_default
 create mode 100644 .config/default_configs.db
 create mode 100644 .config/gce
 create mode 100644 .config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
 create mode 100644 .config/logs/2026.04.02/13.30.17.544197.log
 create mode 100644 .config/logs/2026.04.02/13.30.40.372331.log
 create mode 100644 .config/logs/2026.04.02/13.30.51.422062.log
 create mode 100644 .config/logs/2026.04.02/13.30.52.812826.log
 create mode 100644 .config/logs/2026.04.02/13.31.06.236912.log
 create mode 100644 .config/logs/2026.04.02/13.31.07.077226.log
 create mode 100755 sample_data/README.m

In [34]:
#@title 5. Add your GitHub repository as a remote origin
# Replace YOUR_GITHUB_REPOSITORY_URL with the actual URL of your GitHub repo
# Example: https://github.com/your-username/your-repo-name.git
# If you've already run this, you might need to run !git remote set-url origin YOUR_GITHUB_REPOSITORY_URL instead
GITHUB_REPO_URL = "https://github.com/surajk003/-NLP-Preprocessing-and-Text-Classification" #@param {type:"string"}
!git remote add origin {GITHUB_REPO_URL}
print(f"Remote origin added: {GITHUB_REPO_URL}")

error: remote origin already exists.
Remote origin added: https://github.com/surajk003/-NLP-Preprocessing-and-Text-Classification


In [35]:
from google.colab import userdata

GITHUB_USERNAME = "surajk003" #@param {type:"string"}
# Ensure you have stored your GitHub PAT in Colab Secrets as 'GITHUB_TOKEN'
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

if GITHUB_USERNAME and GITHUB_TOKEN:
    print("Attempting to push to GitHub using PAT...")
    # Construct the push URL with the token
    push_url = f"https://{GITHUB_USERNAME}:{GITHUB_TOKEN}@{GITHUB_REPO_URL.split('https://')[-1]}"
    !git push {push_url} master
    print("Project pushed to GitHub.")
else:
    print("Please provide your GitHub username and ensure GITHUB_TOKEN is set in Colab secrets to push.")
    print("You can also try: !git push -u origin master and enter credentials manually.")

Attempting to push to GitHub using PAT...
Enumerating objects: 28, done.
Counting objects: 100% (28/28), done.
Delta compression using up to 2 threads
Compressing objects: 100% (21/21), done.
Writing objects: 100% (28/28), 8.42 MiB | 1.73 MiB/s, done.
Total 28 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), done.
remote: 
remote: Create a pull request for 'master' on GitHub by visiting:
remote:      https://github.com/surajk003/-NLP-Preprocessing-and-Text-Classification/pull/new/master
remote: 
To https://github.com/surajk003/-NLP-Preprocessing-and-Text-Classification
 * [new branch]      master -> master
Project pushed to GitHub.


After these steps, your Colab project files should be visible in your GitHub repository. Remember to refresh your GitHub repository page to see the changes.

In [23]:
from sklearn.metrics import accuracy_score, classification_report

# Make predictions on the test set
y_pred = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Generate a classification report
report = classification_report(y_test, y_pred, target_names=target_names)
print("\nClassification Report:\n", report)

Accuracy: 0.9161

Classification Report:
                         precision    recall  f1-score   support

           alt.atheism       0.92      0.81      0.86       319
         comp.graphics       0.92      0.96      0.94       389
               sci.med       0.94      0.92      0.93       396
soc.religion.christian       0.88      0.95      0.92       398

              accuracy                           0.92      1502
             macro avg       0.92      0.91      0.91      1502
          weighted avg       0.92      0.92      0.92      1502

